# Proyek Klasifikasi Gambar: EuroSAT RGB
- **Nama:** [Input Nama]
- **Email:** [Input Email]
- **ID Dicoding:** [Input Username]

## Import Semua Packages/Library yang Digunakan

Cell ini memuat library yang dibutuhkan untuk persiapan dataset, mulai dari pembacaan TFDS, manipulasi file, validasi gambar, sampai penyimpanan ringkasan audit.

In [ ]:
import hashlib
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_datasets as tfds
from PIL import Image, UnidentifiedImageError

## Data Preparation

### Setup Reproducibility dan Path

Seed dibuat tetap di angka `42` agar proses split dataset dapat direproduksi. Semua path menggunakan path relatif supaya notebook dapat dijalankan di environment lain tanpa bergantung pada lokasi lokal pribadi.

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

DATASET_NAME = "eurosat/rgb"
DATASET_DIR = Path("dataset")
RAW_DIR = DATASET_DIR / "raw"
TRAIN_DIR = DATASET_DIR / "train"
VALIDATION_DIR = DATASET_DIR / "validation"
TEST_DIR = DATASET_DIR / "test"
AUDIT_DIR = Path("outputs") / "dataset_audit"
TFDS_DATA_DIR = Path("tfds_data")

for directory in [RAW_DIR, TRAIN_DIR, VALIDATION_DIR, TEST_DIR, AUDIT_DIR, TFDS_DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

paths = {
    "raw": str(RAW_DIR),
    "train": str(TRAIN_DIR),
    "validation": str(VALIDATION_DIR),
    "test": str(TEST_DIR),
    "dataset_audit": str(AUDIT_DIR),
    "tfds_data": str(TFDS_DATA_DIR),
}

paths

### Data Loading

Dataset `eurosat/rgb` dimuat dari TensorFlow Datasets. Metadata dataset dipakai untuk mengambil nama kelas, jumlah kelas, split awal, shape gambar, dan dtype tanpa menuliskan daftar kelas secara manual.

In [ ]:
# Download/load TFDS dapat memakan waktu pada eksekusi pertama.
dataset, dataset_info = tfds.load(
    DATASET_NAME,
    split="train",
    as_supervised=True,
    with_info=True,
    shuffle_files=False,
    data_dir=str(TFDS_DATA_DIR),
)

label_feature = dataset_info.features["label"]
image_feature = dataset_info.features["image"]
class_names = list(label_feature.names)
initial_splits = {
    split_name: int(split_info.num_examples)
    for split_name, split_info in dataset_info.splits.items()
}

sample_image, sample_label = next(iter(tfds.as_numpy(dataset.take(1))))

metadata_summary = {
    "dataset_name": DATASET_NAME,
    "total_images": int(dataset_info.splits["train"].num_examples),
    "num_classes": int(label_feature.num_classes),
    "class_names": class_names,
    "initial_splits": initial_splits,
    "image_shape_from_metadata": tuple(image_feature.shape),
    "image_dtype_from_metadata": str(image_feature.dtype),
    "sample_image_shape": tuple(sample_image.shape),
    "sample_image_dtype": str(sample_image.dtype),
    "sample_label": int(sample_label),
}

metadata_summary

### Export TFDS ke Folder Raw

Data dari TFDS diekspor ke `dataset/raw/<class_name>/` dalam format PNG. Struktur folder per kelas memudahkan proses audit, split stratified, dan penggunaan ulang pada tahap modelling berikutnya.

In [ ]:
def reset_directory(directory: Path) -> None:
    if directory.exists():
        shutil.rmtree(directory)
    directory.mkdir(parents=True, exist_ok=True)


for directory in [RAW_DIR, TRAIN_DIR, VALIDATION_DIR, TEST_DIR]:
    reset_directory(directory)

for class_name in class_names:
    (RAW_DIR / class_name).mkdir(parents=True, exist_ok=True)

raw_records = []
raw_class_counter = Counter()

# Proses export seluruh gambar dapat memakan waktu beberapa menit.
for image_index, (image, label) in enumerate(tfds.as_numpy(dataset)):
    label_id = int(label)
    class_name = class_names[label_id]
    raw_class_counter[class_name] += 1

    file_name = f"{class_name}_{raw_class_counter[class_name]:05d}.png"
    image_path = RAW_DIR / class_name / file_name
    Image.fromarray(image).save(image_path, format="PNG")

    raw_records.append(
        {
            "image_index": image_index,
            "label_id": label_id,
            "class_name": class_name,
            "raw_path": str(image_path),
        }
    )

raw_records_df = pd.DataFrame(raw_records)
raw_distribution_df = (
    raw_records_df.groupby("class_name")
    .size()
    .reindex(class_names)
    .reset_index(name="raw_count")
)

raw_distribution_df

### Data Preprocessing

#### Split Dataset

Dataset dibagi secara stratified per kelas menjadi 80% train, 10% validation, dan 10% test. Test set hanya disiapkan untuk evaluasi akhir dan tidak digunakan untuk training maupun tuning.

In [ ]:
split_dirs = {
    "train": TRAIN_DIR,
    "validation": VALIDATION_DIR,
    "test": TEST_DIR,
}

for split_dir in split_dirs.values():
    for class_name in class_names:
        (split_dir / class_name).mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(SEED)
split_records = []

for class_name in class_names:
    class_files = sorted((RAW_DIR / class_name).glob("*.png"))
    class_files = np.array(class_files, dtype=object)
    rng.shuffle(class_files)

    total_class_images = len(class_files)
    train_count = int(total_class_images * 0.8)
    validation_count = int(total_class_images * 0.1)

    split_file_map = {
        "train": class_files[:train_count],
        "validation": class_files[train_count : train_count + validation_count],
        "test": class_files[train_count + validation_count :],
    }

    for split_name, files in split_file_map.items():
        for source_path in files:
            target_path = split_dirs[split_name] / class_name / source_path.name
            shutil.copy2(source_path, target_path)
            split_records.append(
                {
                    "class_name": class_name,
                    "split": split_name,
                    "source_path": str(source_path),
                    "target_path": str(target_path),
                }
            )

split_records_df = pd.DataFrame(split_records)
split_distribution_df = (
    split_records_df.groupby(["class_name", "split"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=class_names, columns=["train", "validation", "test"], fill_value=0)
    .reset_index()
)

dataset_split_summary_df = raw_distribution_df.merge(
    split_distribution_df,
    on="class_name",
    how="left",
)
dataset_split_summary_df["total_after_split"] = dataset_split_summary_df[
    ["train", "validation", "test"]
].sum(axis=1)

dataset_split_summary_path = AUDIT_DIR / "dataset_split_summary.csv"
dataset_split_summary_df.to_csv(dataset_split_summary_path, index=False)

dataset_split_summary_df

### Audit Dataset

Audit dilakukan untuk memeriksa distribusi kelas sebelum dan sesudah split, resolusi gambar, mode warna, format file, gambar corrupt, duplikasi file dalam setiap folder, serta duplikasi antar split train/validation/test.

In [ ]:
def calculate_sha256(file_path: Path, chunk_size: int = 1024 * 1024) -> str:
    sha256_hash = hashlib.sha256()
    with file_path.open("rb") as file:
        for chunk in iter(lambda: file.read(chunk_size), b""):
            sha256_hash.update(chunk)
    return sha256_hash.hexdigest()


def inspect_image(file_path: Path) -> dict:
    try:
        with Image.open(file_path) as image:
            resolution = f"{image.width}x{image.height}"
            mode = image.mode
            image_format = image.format
            image.verify()

        return {
            "is_corrupt": False,
            "resolution": resolution,
            "mode": mode,
            "format": image_format,
            "sha256": calculate_sha256(file_path),
            "error": None,
        }
    except (UnidentifiedImageError, OSError, ValueError) as error:
        return {
            "is_corrupt": True,
            "resolution": None,
            "mode": None,
            "format": None,
            "sha256": None,
            "error": str(error),
        }


def collect_image_audit(root_dir: Path, dataset_part: str) -> list:
    audit_rows = []
    for file_path in sorted(root_dir.rglob("*")):
        if not file_path.is_file():
            continue

        image_info = inspect_image(file_path)
        audit_rows.append(
            {
                "dataset_part": dataset_part,
                "class_name": file_path.parent.name,
                "path": str(file_path),
                **image_info,
            }
        )
    return audit_rows


# Audit seluruh file gambar dapat memakan waktu karena setiap file dibuka dan dihitung hash-nya.
audit_rows = []
audit_roots = {
    "raw": RAW_DIR,
    "train": TRAIN_DIR,
    "validation": VALIDATION_DIR,
    "test": TEST_DIR,
}

for dataset_part, root_dir in audit_roots.items():
    audit_rows.extend(collect_image_audit(root_dir, dataset_part))

audit_df = pd.DataFrame(audit_rows)
valid_audit_df = audit_df[~audit_df["is_corrupt"]].copy()

def value_counts_dict(series: pd.Series) -> dict:
    return {str(key): int(value) for key, value in series.value_counts(dropna=False).sort_index().items()}


duplicate_files_within_parts = []
for dataset_part, part_df in valid_audit_df.groupby("dataset_part"):
    duplicate_hashes = part_df[part_df.duplicated("sha256", keep=False)]
    for file_hash, hash_df in duplicate_hashes.groupby("sha256"):
        duplicate_files_within_parts.append(
            {
                "dataset_part": dataset_part,
                "sha256": file_hash,
                "count": int(len(hash_df)),
                "paths": sorted(hash_df["path"].tolist()),
            }
        )

split_only_df = valid_audit_df[valid_audit_df["dataset_part"].isin(["train", "validation", "test"])]
duplicates_across_splits = []
for file_hash, hash_df in split_only_df.groupby("sha256"):
    split_names = sorted(hash_df["dataset_part"].unique().tolist())
    if len(split_names) > 1:
        duplicates_across_splits.append(
            {
                "sha256": file_hash,
                "splits": split_names,
                "paths": sorted(hash_df["path"].tolist()),
            }
        )

corrupt_images = audit_df[audit_df["is_corrupt"]][["dataset_part", "class_name", "path", "error"]]

dataset_audit_summary = {
    "dataset_name": DATASET_NAME,
    "seed": SEED,
    "paths": paths,
    "metadata": metadata_summary,
    "distribution_before_split": raw_distribution_df.to_dict(orient="records"),
    "distribution_after_split": dataset_split_summary_df.to_dict(orient="records"),
    "audit": {
        "total_files_checked": int(len(audit_df)),
        "total_valid_images": int(len(valid_audit_df)),
        "total_corrupt_images": int(len(corrupt_images)),
        "corrupt_images": corrupt_images.to_dict(orient="records"),
        "resolution_distribution": value_counts_dict(valid_audit_df["resolution"]),
        "mode_distribution": value_counts_dict(valid_audit_df["mode"]),
        "format_distribution": value_counts_dict(valid_audit_df["format"]),
        "duplicate_files_within_parts_count": int(len(duplicate_files_within_parts)),
        "duplicate_files_within_parts": duplicate_files_within_parts,
        "duplicates_across_train_validation_test_count": int(len(duplicates_across_splits)),
        "duplicates_across_train_validation_test": duplicates_across_splits,
    },
}

dataset_audit_summary_path = AUDIT_DIR / "dataset_audit_summary.json"
with dataset_audit_summary_path.open("w", encoding="utf-8") as file:
    json.dump(dataset_audit_summary, file, indent=2)

{
    "split_summary_csv": str(dataset_split_summary_path),
    "audit_summary_json": str(dataset_audit_summary_path),
    "total_corrupt_images": dataset_audit_summary["audit"]["total_corrupt_images"],
    "duplicate_files_within_parts_count": dataset_audit_summary["audit"]["duplicate_files_within_parts_count"],
    "duplicates_across_train_validation_test_count": dataset_audit_summary["audit"]["duplicates_across_train_validation_test_count"],
}

## Modelling

## Evaluasi dan Visualisasi

## Konversi Model

## Inference (Optional)